# Section 5: RQ3 — DBS Classification & Cross-Condition — 10 figures

| # | Figure | Count |
|---|--------|-------|
| 49 | Classification grouped bar chart | 1 |
| 50 | Standard classification heatmap | 1 |
| 51 | Flipped classification heatmap | 1 |
| 52 | ROC curves | 1 |
| 53-54 | Within vs cross-condition RMSE | 2 |
| 55-56 | Cross-block predictions | 2 |
| 57-58 | Forecast checkpoint comparison | 2 |

In [ ]:
import sys, os

os.chdir("/home/bobby/repos/latent-neural-dynamics-modeling")
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")

from pathlib import Path
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix as sk_confusion_matrix, balanced_accuracy_score

from modules.style import (
    COLOR_CHANCE,
    COLOR_DBS_OFF,
    COLOR_DBS_ON,
    COLOR_DPAD,
    COLOR_NS,
    COLOR_PSID,
    COLOR_VARMA,
    apply_modules.style,
    dbs_color,
    hex_to_rgba,
    panel_label,
    stack_bar_label,
)
from modules.loaders import discover_session_run, EXP_BEHAVIORAL, EXP_NEURAL, SESSIONS

apply_modules.style()

OUT = Path("thesis_figures/sec5")
OUT.mkdir(parents=True, exist_ok=True)
results_root = Path("results").resolve()
from types import SimpleNamespace as _NS


def _session_ns(session, exp_type):
    pv, pt = discover_session_run(results_root, "psid", exp_type, session)
    vv, vt = discover_session_run(results_root, "varma", exp_type, session)
    dv, dt = discover_session_run(results_root, "dpad", exp_type, session)
    return _NS(
        psid_variant=pv or "", psid_run_ts=pt or "",
        varma_variant=vv or "", varma_run_ts=vt or "",
        dpad_variant=dv or "", dpad_run_ts=dt or "", label=session,
    )


_all_session_objs = [_session_ns(s, EXP_BEHAVIORAL) for s in SESSIONS]
_all_lap_session_objs = [_session_ns(s, EXP_NEURAL) for s in SESSIONS]

COLOR_TRUE = "#1A1A1A"


## Classification spec lists

In [ ]:
GROUP_ORDER: Tuple[str, ...] = ("xp", "xp_1", "xp_2", "xp_with_dbs")
GROUP_SHORT: Dict[str, str] = {
    "xp": "Xp",
    "xp_1": "Xp1",
    "xp_2": "Xp2",
    "xp_with_dbs": "Xp+DBS",
}
GROUP_COLORS: Dict[str, str] = {
    "xp": COLOR_PSID,
    "xp_1": "#0F6E56",
    "xp_2": "#993C1D",
    "xp_with_dbs": "#854F0B",
}

# Canonical ordered session keys — behavioral first, laplacian second.
_ORDERED_SESSION_KEYS: List[str] = [
    "PDI1_S2",
    "PDI1_S4",
    "PDI4_S2",
    "PDI4_S3",
    "PDI1_S2 (lap)",
    "PDI1_S4 (lap)",
    "PDI4_S2 (lap)",
    "PDI4_S3 (lap)",
]


@dataclass(frozen=True)
class ClassificationF1Point:
    participant_label: str
    session_label: str
    group: str
    balanced_accuracy: float
    permutation_pvalue: Optional[float]
    model_label: str = "PSID"
    permutation_scores: Optional[Tuple[float, ...]] = None


# ── Parquet loaders ──────────────────────────────────────────────────────────

_PRED_SRC_TO_GROUP: Dict[str, str] = {
    "Xp": "xp",
    "Xp_1": "xp_1",
    "Xp_2": "xp_2",
    "Xp_with_dbs": "xp_with_dbs",
}
_FCST_SRC_TO_GROUP: Dict[str, str] = {
    "Xf": "xp",
    "Xf_1": "xp_1",
    "Xf_2": "xp_2",
    "Xf_with_dbs": "xp_with_dbs",
}


def _load_sweep_df(results_root: Path) -> pd.DataFrame:
    files = glob.glob(
        str(results_root / "psid" / "*" / "classification" / "sweep_*.parquet")
    )
    if not files:
        return pd.DataFrame()
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    return df.sort_values("run_ts").drop_duplicates(
        subset=["variant", "sub_source", "flipped", "mode"], keep="last"
    )


def _parse_variant_meta(
    variant: str,
) -> Tuple[Optional[str], Optional[str], Optional[str], Optional[int]]:
    """Returns (exp_type, participant, session, n1) or (None, None, None, None)."""
    m = re.search(r"z-as-(behavior|neural)_(PDI\d+)_(S\d+).*?_n1_(\d+)", variant)
    if not m:
        return None, None, None, None
    exp_type = "behavioral" if m.group(1) == "behavior" else "laplacian"
    return exp_type, m.group(2), m.group(3), int(m.group(4))


_sweep_df = _load_sweep_df(results_root)


def build_cls_points_from_parquet(
    mode: str = "predictions",
    flipped: bool = False,
    dbs_filter: str = "both",
) -> List[ClassificationF1Point]:
    """One ClassificationF1Point per (session, feature group) from sweep parquets."""
    src_map = _PRED_SRC_TO_GROUP if mode == "predictions" else _FCST_SRC_TO_GROUP
    if _sweep_df.empty:
        return []
    df = _sweep_df[
        (_sweep_df["mode"] == mode)
        & (_sweep_df["flipped"] == flipped)
        & (_sweep_df["sub_source"].isin(src_map.keys()))
        & (_sweep_df["variant"].str.contains(f"_dbs_{dbs_filter}$"))
    ]
    # For forecast-flipped: pick best cv_ba per (variant, sub_source).
    if flipped:
        df = df.sort_values("cv_ba", ascending=False).drop_duplicates(
            subset=["variant", "sub_source"]
        )
    points: List[ClassificationF1Point] = []
    for _, row in df.iterrows():
        exp_type, participant, session, _ = _parse_variant_meta(row["variant"])
        if participant is None:
            continue
        group = src_map[row["sub_source"]]
        sess_label = session if exp_type == "behavioral" else f"{session} (lap)"
        pval = float(row["p_value"]) if pd.notna(row["p_value"]) else None
        points.append(
            ClassificationF1Point(
                participant_label=participant,
                session_label=sess_label,
                group=group,
                balanced_accuracy=float(row["cv_ba"]),
                permutation_pvalue=pval,
                model_label="PSID",
            )
        )
    return points


def build_forecast_bests_from_parquet() -> (
    Dict[Tuple[str, str], Tuple[float, Optional[str], Optional[float], None]]
):
    """Best-cv_ba (h, m) per (session_key, feature_group) for forecast classification."""
    if _sweep_df.empty:
        return {}
    df = _sweep_df[
        (_sweep_df["mode"] == "forecast")
        & (_sweep_df["flipped"] == False)
        & (_sweep_df["sub_source"].isin(_FCST_SRC_TO_GROUP.keys()))
        & (_sweep_df["variant"].str.contains("_dbs_both$"))
    ]
    best = df.sort_values("cv_ba", ascending=False).drop_duplicates(
        subset=["variant", "sub_source"]
    )
    out: Dict[Tuple[str, str], Tuple[float, Optional[str], Optional[float], None]] = {}
    for _, row in best.iterrows():
        exp_type, participant, session, _ = _parse_variant_meta(row["variant"])
        if participant is None:
            continue
        group = _FCST_SRC_TO_GROUP[row["sub_source"]]
        sess_label = session if exp_type == "behavioral" else f"{session} (lap)"
        sess_key = f"{participant}_{sess_label}"
        pval = float(row["p_value"]) if pd.notna(row["p_value"]) else None
        h_m = f"h{row['h_seconds']:g}_m{row['m_test_seconds']:g}"
        out[(sess_key, group)] = (float(row["cv_ba"]), h_m, pval, None)
    return out


def build_ba_pv_matrix(
    points: List[ClassificationF1Point],
    session_keys: List[str],
    feat_keys: List[str],
) -> Tuple[np.ndarray, np.ndarray]:
    """Build (BA, p_value) matrices from a list of ClassificationF1Points."""
    n_rows, n_cols = len(session_keys), len(feat_keys)
    ba = np.full((n_rows, n_cols), np.nan)
    pv = np.full((n_rows, n_cols), np.nan)
    for pt in points:
        sk = f"{pt.participant_label}_{pt.session_label}"
        if sk not in session_keys or pt.group not in feat_keys:
            continue
        r, c = session_keys.index(sk), feat_keys.index(pt.group)
        if np.isnan(ba[r, c]) or pt.balanced_accuracy > ba[r, c]:
            ba[r, c] = pt.balanced_accuracy
            pv[r, c] = (
                pt.permutation_pvalue if pt.permutation_pvalue is not None else np.nan
            )
    return ba, pv


# ── Inline LDA confusion matrix (test split) ─────────────────────────────────


def _xp_mean_features(df_split: pd.DataFrame, n1: int, sub_source: str) -> np.ndarray:
    """Mean of Xp over time per trial; slice to sub_source feature subset. Returns (n_trials, n_features)."""

    def trial_mean(xp_raw) -> np.ndarray:
        return np.stack([np.asarray(x) for x in xp_raw]).mean(axis=0)  # (nx,)

    X = np.array([trial_mean(df_split["Xp"].iloc[i]) for i in range(len(df_split))])
    if sub_source == "Xp_1":
        return X[:, :n1]
    if sub_source == "Xp_2":
        return X[:, n1:]
    return X


def build_confusion_matrix(
    variant: str, sub_source: str, results_root: Path
) -> Optional[np.ndarray]:
    """Fit LDA on mean-Xp from train+val; predict on test. Returns 2x2 cm or None."""
    _, _, _, n1 = _parse_variant_meta(variant)
    if n1 is None:
        return None

    def load_split(split: str) -> Optional[pd.DataFrame]:
        split_dir = results_root / "psid" / variant / "inference" / split
        if not split_dir.exists():
            return None
        files = sorted(split_dir.glob("*.parquet"))
        return pd.read_parquet(files[-1]) if files else None

    df_train = load_split("train")
    df_test = load_split("test")
    if df_train is None or df_test is None:
        return None

    def Xy(df: pd.DataFrame):
        X = _xp_mean_features(df, n1, sub_source)
        y = (df["stim"].values == "on").astype(int)
        return X, y

    X_tr, y_tr = Xy(df_train)
    X_te, y_te = Xy(df_test)
    df_val = load_split("val")
    if df_val is not None:
        X_vl, y_vl = Xy(df_val)
        X_pool = np.vstack([X_tr, X_vl])
        y_pool = np.concatenate([y_tr, y_vl])
    else:
        X_pool, y_pool = X_tr, y_tr

    if X_pool.shape[1] == 0 or len(np.unique(y_pool)) < 2:
        return None
    try:
        lda = LinearDiscriminantAnalysis()
        lda.fit(X_pool, y_pool)
        y_pred = lda.predict(X_te)
        return sk_confusion_matrix(y_te, y_pred, labels=[0, 1])
    except Exception:
        return None


print(
    f"Sweep parquet: {len(_sweep_df)} rows, {_sweep_df['variant'].nunique()} variants."
)
print(f"Ordered session keys: {_ORDERED_SESSION_KEYS}")

## Fig 49: Classification — prediction vs forecast (2x1 pooled box + strip)

**Decoupled PSID vs DPAD.** PSID is the focus; DPAD classification phase 5 is
being re-run (yesterday's 2026-04-27 chain crashed at Step 2 perm). Once the
DPAD pickles ship `permutation_test["scores"]`, set `SHOW_DPAD = True` to
overlay a second model.

**Chance level.** Balanced-accuracy chance is *not* 0.5 here. Per-cell
permutation null distributions sit well above 0.5 because (a) `Xp_with_dbs`
encodes DBS state directly, and each block is pure on/off, so a permuted-label
LDA still recovers block from the DBS column (perm 95% / max → 1.0); (b)
chrono-block CV with few groups limits the entropy of the label permutation.
We replace the single 0.5 line with a per-`(feature, mode)` empirical chance
band — 5–95% of the pooled permutation distribution across cells that ship
full `scores`.

In [ ]:
import csv
from matplotlib.patches import Patch as _Patch

SHOW_DPAD = False

cls_points = build_cls_points_from_parquet(
    mode="predictions", flipped=False, dbs_filter="both"
)
forecast_bests_psid = build_forecast_bests_from_parquet()
forecast_bests_dpad: Dict[Tuple[str, str], Any] = {}


def _mode_from_session_label(session_label: str) -> str:
    return "laplacian" if "(lap)" in session_label else "behavioral"


# Pooled buckets keyed by (feature, model, mode).
pooled_pred: Dict[Tuple[str, str, str], List] = {}
for pt in cls_points:
    sess_key = f"{pt.participant_label}_{pt.session_label}"
    mode = _mode_from_session_label(pt.session_label)
    pooled_pred.setdefault((pt.group, pt.model_label, mode), []).append(
        (sess_key, pt.balanced_accuracy, pt.permutation_pvalue, None)
    )

pooled_fcst: Dict[Tuple[str, str, str], List] = {}
for src, model in [(forecast_bests_psid, "PSID"), (forecast_bests_dpad, "DPAD")]:
    for (sess, grp), (ba, hm, pval, _pscores) in src.items():
        mode = _mode_from_session_label(sess)
        pooled_fcst.setdefault((grp, model, mode), []).append(
            (sess, ba, pval, None, hm)
        )

# Session keys in canonical order.
_all_skeys = {f"{pt.participant_label}_{pt.session_label}" for pt in cls_points}
session_keys = [sk for sk in _ORDERED_SESSION_KEYS if sk in _all_skeys]

pred_lookup: Dict[Tuple[str, str, str], ClassificationF1Point] = {
    (f"{pt.participant_label}_{pt.session_label}", pt.group, pt.model_label): pt
    for pt in cls_points
}

features = list(GROUP_ORDER)


def _model_color(model: str) -> str:
    return COLOR_PSID if model == "PSID" else COLOR_DPAD


def _draw_box(
    ax, vals, x_pos, color, *, hatch=None, fill_alpha=0.25, width: float = 0.14
):
    face = (*to_rgba(color)[:3], fill_alpha)
    bp = ax.boxplot(
        [vals],
        positions=[x_pos],
        widths=width,
        patch_artist=True,
        showfliers=False,
        manage_ticks=False,
    )
    for box in bp["boxes"]:
        box.set(facecolor=face, edgecolor=color, linewidth=1.0)
        if hatch:
            box.set_hatch(hatch)
    for elem in ("whiskers", "caps", "medians"):
        for ln in bp[elem]:
            ln.set(color=color, linewidth=1.0)
    return bp


def _hm_short(hm: Optional[str]) -> str:
    if not hm:
        return "-"
    parts = hm.split("_")
    if len(parts) != 2 or not parts[0].startswith("h") or not parts[1].startswith("m"):
        return hm
    return f"{parts[0][1:]}/{parts[1][1:]}"


def render_pooled_classification(
    model_label: str, save_path: Path, *, title_suffix: str = ""
) -> bool:
    mm_groups = [(model_label, mo) for mo in ("behavioral", "laplacian")]
    box_spread, box_width = 0.10, 0.08
    mm_offsets = np.array([-box_spread, +box_spread])

    has_any = False
    n_sess = len(session_keys)
    fig, all_axes = plt.subplots(
        3,
        1,
        figsize=(7.5, 7.6 + 0.18 * n_sess),
        gridspec_kw={"height_ratios": [3, 3, max(1.5, 0.18 * n_sess)]},
    )
    axes = all_axes[:2]
    rng = np.random.default_rng(42)
    color = _model_color(model_label)

    for ax, src in zip(axes, [pooled_pred, pooled_fcst]):
        for fi, feat in enumerate(features):
            for gi, (_m, mode) in enumerate(mm_groups):
                entries = src.get((feat, model_label, mode), [])
                x_pos = fi + mm_offsets[gi]
                if not entries:
                    continue
                hatch = "//" if mode == "laplacian" else None
                bas = np.array([e[1] for e in entries], dtype=float)
                bas_finite = bas[np.isfinite(bas)]
                if len(bas_finite) >= 2:
                    _draw_box(
                        ax, bas_finite, x_pos, color, hatch=hatch, width=box_width
                    )
                    has_any = True
                jitter = rng.uniform(-box_width * 0.25, box_width * 0.25, len(entries))
                for j, entry in enumerate(entries):
                    ba, pval = entry[1], entry[2]
                    if not np.isfinite(ba):
                        continue
                    has_any = True
                    sig = pval is not None and pval < 0.05
                    ax.scatter(
                        x_pos + jitter[j],
                        ba,
                        s=28,
                        color=color,
                        alpha=1.0 if (sig or pval is None) else 0.55,
                        edgecolor="black" if (sig or pval is None) else "none",
                        linewidths=0.5,
                        zorder=3,
                    )
        ax.axhline(0.5, linestyle=":", color="#999999", linewidth=0.8)
        ax.set_ylim(0.3, 1.05)
        ax.set_ylabel("Balanced accuracy (CV)")

    axes[1].set_xticks(np.arange(len(features)))
    axes[1].set_xticklabels([GROUP_SHORT[f] for f in features])
    axes[1].set_xlabel("Feature group")
    panel_label(axes[0], "A", f"{model_label} prediction (Xp at t){title_suffix}")
    panel_label(axes[1], "B", f"{model_label} forecast (best h, m){title_suffix}")

    legend_handles = [
        _Patch(
            facecolor=hex_to_rgba(color, 0.25),
            edgecolor=color,
            label=f"{model_label}, behavioral",
        ),
        _Patch(
            facecolor=hex_to_rgba(color, 0.25),
            edgecolor=color,
            hatch="//",
            label=f"{model_label}, laplacian",
        ),
        plt.Line2D(
            [0],
            [0],
            linestyle=":",
            color="#999999",
            linewidth=0.8,
            label="0.5 reference",
        ),
    ]
    axes[0].legend(handles=legend_handles, fontsize=8)

    all_axes[2].axis("off")
    fig.tight_layout()

    if has_any:
        fig.savefig(str(save_path))
        plt.show()
    else:
        plt.close(fig)
    return has_any


render_pooled_classification("PSID", OUT / "fig_049_classification_pooled_psid.png")
_dpad_rendered = False

# Appendix CSV.
table_path = OUT / "table_049_classification.csv"
with open(table_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(
        [
            "panel",
            "session",
            "feature",
            "model",
            "mode",
            "balanced_accuracy",
            "perm_pvalue",
            "best_h_m",
        ]
    )
    for pt in cls_points:
        sess_key = f"{pt.participant_label}_{pt.session_label}"
        mode = _mode_from_session_label(pt.session_label)
        w.writerow(
            [
                "prediction",
                sess_key,
                pt.group,
                pt.model_label,
                mode,
                (
                    f"{pt.balanced_accuracy:.4f}"
                    if np.isfinite(pt.balanced_accuracy)
                    else ""
                ),
                (
                    f"{pt.permutation_pvalue:.4f}"
                    if pt.permutation_pvalue is not None
                    else ""
                ),
                "",
            ]
        )
    for src_dict, model in [
        (forecast_bests_psid, "PSID"),
        (forecast_bests_dpad, "DPAD"),
    ]:
        for (sess, grp), (ba, hm, pval, _pscores) in src_dict.items():
            mode = _mode_from_session_label(sess)
            w.writerow(
                [
                    "forecast",
                    sess,
                    grp,
                    model,
                    mode,
                    f"{ba:.4f}" if np.isfinite(ba) else "",
                    f"{pval:.4f}" if pval is not None else "",
                    hm or "",
                ]
            )

print(
    f"Fig 49 - PSID classification (cv_ba). Panel A: prediction Xp. Panel B: forecast best (h,m). "
    f"Dots: full + edge = p<0.05, faded = not significant, full + edge = no perm test. "
    f"Behavioral + laplacian modes shown per feature group. Sessions: {len(session_keys)}. "
    f"CSV: {table_path.name}"
)

In [ ]:
# check the permutation distribution average. (close to be 0.5)
# fishy about the classification. (fishy).

## Fig 49a: Confusion matrix grid — sessions × features (PSID)

8 sessions × 4 feature groups = 32 small 2×2 confusion matrices, row-normalized
(rows = true class). Diagonal = TPR(off) and TPR(on). Below-cell text shows the
raw counts. PSID-only when `SHOW_DPAD=False`; when DPAD ships, two grids
stacked vertically.

In [ ]:
def _draw_confusion_cell(ax, cm: Optional[np.ndarray], *, title: str, color: str):
    if cm is None:
        ax.add_patch(
            plt.Rectangle(
                (0, 0),
                1,
                1,
                facecolor="#EEEEEE",
                edgecolor="#BBBBBB",
                hatch="///",
                linewidth=0.5,
            )
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(title, fontsize=7)
        return
    # Reorient: sklearn order [off=0, on=1] -> flip so on is top row.
    cm_oriented = cm[::-1, ::-1]
    row_sums = cm_oriented.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    norm_cm = cm_oriented / row_sums
    base_rgba = np.asarray(to_rgba(color))
    rgba = np.zeros((2, 2, 4))
    for i in range(2):
        for j in range(2):
            a = float(np.clip(norm_cm[i, j], 0, 1))
            rgba[i, j, :3] = base_rgba[:3]
            rgba[i, j, 3] = 0.15 + 0.75 * a
    ax.imshow(rgba, aspect="equal", interpolation="nearest")
    for i in range(2):
        for j in range(2):
            txt_color = "white" if norm_cm[i, j] > 0.55 else "#222222"
            ax.text(
                j,
                i,
                f"{norm_cm[i, j]:.2f}\n({int(cm_oriented[i, j])})",
                ha="center",
                va="center",
                fontsize=6,
                color=txt_color,
            )
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["on", "off"], fontsize=6)
    ax.set_yticklabels(["on", "off"], fontsize=6)
    ax.set_title(title, fontsize=7)


def _confusion_grid_psid() -> Optional[plt.Figure]:
    """Build confusion matrix grid via inline LDA refit from inference splits."""
    # For each (session, feature): find the dbs_both variant and refit LDA.
    # Map session_key -> variant name.
    variant_map: Dict[str, str] = {}
    for v in _sweep_df["variant"].unique():
        exp_type, participant, session, _ = _parse_variant_meta(v)
        if participant is None or not v.endswith("_dbs_both"):
            continue
        sess_label = session if exp_type == "behavioral" else f"{session} (lap)"
        sk = f"{participant}_{sess_label}"
        variant_map[sk] = v

    # Sub-source for each feature group (predictions -> Xp family).
    group_to_src = {"xp": "Xp", "xp_1": "Xp_1", "xp_2": "Xp_2", "xp_with_dbs": "Xp"}

    n_rows, n_cols = len(session_keys), len(features)
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(1.4 * n_cols + 0.6, 1.4 * n_rows + 0.6)
    )
    axes = np.atleast_2d(axes)
    drew_any = False

    for ri, sk in enumerate(session_keys):
        variant = variant_map.get(sk)
        for ci, feat in enumerate(features):
            ax = axes[ri, ci]
            cm = None
            if variant:
                sub_src = group_to_src[feat]
                cm = build_confusion_matrix(variant, sub_src, results_root)
            title = f"{GROUP_SHORT[feat]}" if ri == 0 else ""
            _draw_confusion_cell(ax, cm, title=title, color=COLOR_PSID)
            if cm is not None:
                drew_any = True
        axes[ri, 0].set_ylabel(sk, fontsize=7)

    fig.suptitle(
        "PSID - confusion matrices (test split, LDA on train+val mean Xp)",
        fontsize=9,
        y=0.995,
    )
    fig.tight_layout()
    return fig if drew_any else None


fig_psid = _confusion_grid_psid()
if fig_psid is not None:
    fig_psid.savefig(str(OUT / "fig_049a_confusion_psid.png"))
    plt.show()
    print(
        f"Fig 49a - PSID confusion matrices (test split). {len(session_keys)} sessions x {len(features)} features. "
        "Row-normalized, on=top row. Inline LDA refit on mean-Xp features from train+val."
    )
else:
    print("Fig 49a SKIPPED - no confusion data (check inference splits exist).")

## Fig 50: Standard classification heatmap (PSID + DPAD stacked)

In [ ]:
from matplotlib.colors import TwoSlopeNorm

feat_keys = list(GROUP_ORDER)
n_rows_hm = len(session_keys)
n_cols_hm = len(feat_keys)

psid_ba, psid_pv = build_ba_pv_matrix(cls_points, session_keys, feat_keys)
dpad_ba = np.full((n_rows_hm, n_cols_hm), np.nan)
dpad_pv = np.full((n_rows_hm, n_cols_hm), np.nan)

cmap = plt.cm.RdBu_r
norm = TwoSlopeNorm(vmin=0.4, vcenter=0.5, vmax=0.7)


def _draw_heatmap(ax, mat: np.ndarray, pv: np.ndarray, *, title: str):
    rgba = cmap(norm(np.clip(mat, 0.4, 0.7)))
    alpha = np.where(np.isfinite(pv) & (pv < 0.05), 1.0, 0.65)
    alpha = np.where(np.isfinite(mat), alpha, 0.0)
    rgba[..., -1] = alpha
    ax.imshow(rgba, aspect="auto", interpolation="nearest")
    for ri in range(n_rows_hm):
        for ci in range(n_cols_hm):
            if not np.isfinite(mat[ri, ci]):
                ax.add_patch(
                    plt.Rectangle(
                        (ci - 0.5, ri - 0.5),
                        1,
                        1,
                        facecolor="#EEEEEE",
                        edgecolor="#BBBBBB",
                        hatch="///",
                        linewidth=0.5,
                    )
                )
            else:
                v = mat[ri, ci]
                ax.text(
                    ci,
                    ri,
                    f"{v:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white" if abs(v - 0.5) > 0.1 else "#222222",
                )
    ax.set_xticks(np.arange(n_cols_hm))
    ax.set_xticklabels([GROUP_SHORT[f] for f in feat_keys])
    ax.set_yticks(np.arange(n_rows_hm))
    ax.set_yticklabels(session_keys)
    ax.set_xlabel("Feature group")
    ax.set_ylabel("Session")
    ax.set_title(title, loc="left", fontsize=10)


fig, ax_only = plt.subplots(1, 1, figsize=(7.5, 5.0))
_draw_heatmap(ax_only, psid_ba, psid_pv, title="PSID - prediction cv_ba")

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=ax_only, fraction=0.04, pad=0.02, label="Balanced accuracy (CV)"
)
cbar.ax.axhline(0.5, color="#222222", linewidth=0.8, linestyle="--")

fig.savefig(str(OUT / "fig_050_classification_heatmap.png"))
plt.show()

print(
    "Fig 50 - PSID classification heatmap. Rows: sessions, cols: feature groups. "
    "Color: cv_ba (RdBu_r centred at 0.5). Alpha: full = p<0.05, faded = NS. "
    "Hatched = missing. Numbers inside cells."
)

## Fig 51: Flipped classification heatmap (PSID + DPAD stacked)

In [ ]:
# Flipped = cross-condition forecast classification (mode=forecast, flipped=True).
# Note: flipped=True only exists for forecast mode, not predictions.
flip_points = build_cls_points_from_parquet(
    mode="forecast", flipped=True, dbs_filter="both"
)
psid_flip_ba, psid_flip_pv = build_ba_pv_matrix(flip_points, session_keys, feat_keys)
dpad_flip_ba = np.full((n_rows_hm, n_cols_hm), np.nan)
dpad_flip_pv = np.full((n_rows_hm, n_cols_hm), np.nan)

fig, ax_only = plt.subplots(1, 1, figsize=(7.5, 5.0))
_draw_heatmap(
    ax_only, psid_flip_ba, psid_flip_pv, title="PSID - flipped forecast cv_ba"
)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=ax_only, fraction=0.04, pad=0.02, label="Balanced accuracy (CV)"
)
cbar.ax.axhline(0.5, color="#222222", linewidth=0.8, linestyle="--")

fig.savefig(str(OUT / "fig_051_flipped_heatmap.png"))
plt.show()

print(
    f"Fig 51 - Flipped (cross-condition) forecast classification cv_ba. "
    f"Uses mode=forecast, flipped=True from sweep parquet. "
    f"Best (h,m) per (session, feature). Sessions: {len(session_keys)}."
)

## Fig 52: ROC curves — per session, feature × model overlays

In [ ]:
print(
    "Fig 52 (ROC curves) - SKIPPED. Requires inline LDA refit with predict_proba. "
    "Use build_confusion_matrix() extended with proba output for future implementation."
)

## Fig 52a: Paired Δ scatter — DPAD vs PSID test BA per (cell × feature)

Requires DPAD classification. Gated on SHOW_DPAD; skipped while DPAD runs.

In [ ]:
if not SHOW_DPAD:
    print("Fig 52a SKIPPED — DPAD classification pending (run pipeline_dpad phase 5).")
    paired_count = {"behavioral": 0, "laplacian": 0}
else:
    psid_lookup_paired: Dict[Tuple[str, str, str], ClassificationF1Point] = {
        (
            f"{pt.participant_label}_{pt.session_label}",
            pt.group,
            "behavioral" if "(lap)" not in pt.session_label else "laplacian",
        ): pt
        for pt in cls_points
        if pt.model_label == "PSID"
    }
    dpad_lookup_paired: Dict[Tuple[str, str, str], ClassificationF1Point] = {
        (
            f"{pt.participant_label}_{pt.session_label}",
            pt.group,
            "behavioral" if "(lap)" not in pt.session_label else "laplacian",
        ): pt
        for pt in cls_points
        if pt.model_label == "DPAD"
    }

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.plot(
        [0.3, 1.05],
        [0.3, 1.05],
        linestyle="--",
        color="#555555",
        linewidth=0.9,
        label="y = x",
    )
    ax.axhline(0.5, linestyle=":", color="#888888", linewidth=0.7)
    ax.axvline(0.5, linestyle=":", color="#888888", linewidth=0.7)

    paired_count = {"behavioral": 0, "laplacian": 0}
    for (sk, feat, mode), psid_pt in psid_lookup_paired.items():
        dpad_pt = dpad_lookup_paired.get((sk, feat, mode))
        if dpad_pt is None or not np.isfinite(dpad_pt.balanced_accuracy):
            continue
        if not np.isfinite(psid_pt.balanced_accuracy):
            continue
        color = GROUP_COLORS[feat]
        marker = "o" if mode == "behavioral" else "s"
        sig_psid = (
            psid_pt.permutation_pvalue is not None and psid_pt.permutation_pvalue < 0.05
        )
        sig_dpad = (
            dpad_pt.permutation_pvalue is not None and dpad_pt.permutation_pvalue < 0.05
        )
        a = 1.0 if (sig_psid or sig_dpad) else 0.55
        ax.scatter(
            psid_pt.balanced_accuracy,
            dpad_pt.balanced_accuracy,
            s=55,
            color=color,
            alpha=a,
            marker=marker,
            edgecolor="black" if (sig_psid or sig_dpad) else "none",
            linewidths=0.5,
        )
        paired_count[mode] += 1

    ax.set_xlabel("PSID test BA")
    ax.set_ylabel("DPAD test BA")
    ax.set_xlim(0.3, 1.05)
    ax.set_ylim(0.3, 1.05)
    ax.set_aspect("equal")

    legend_handles = [
        _Patch(
            facecolor=GROUP_COLORS[g], edgecolor=GROUP_COLORS[g], label=GROUP_SHORT[g]
        )
        for g in GROUP_ORDER
    ]
    legend_handles += [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="#555555",
            markersize=8,
            label="behavioral",
        ),
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor="#555555",
            markersize=8,
            label="laplacian",
        ),
    ]
    ax.legend(handles=legend_handles)

    fig.savefig(str(OUT / "fig_052a_paired_ba_scatter.png"))
    plt.show()

    print(
        "Fig 52a - Paired DPAD vs PSID test BA. Each point: one (cell × feature). "
        "Marker shape = mode (circle behavioral, square laplacian); colour = feature "
        "group. Diagonal = parity. Significant points (perm p<0.05 in either model) "
        "shown at full opacity with edge stroke. Empty regions = DPAD classification "
        f"pending. Pairs plotted: {paired_count['behavioral']} behav + {paired_count['laplacian']} lap."
    )